# MePRAM preprocessing overview

This documented copy outlines how raw MePRAM tables are loaded, harmonised, and merged into modelling-ready datasets, highlighting the intent of each processing stage.

## Environment setup

Import the scientific Python stack, profiling utilities, and Spark placeholders required throughout the preprocessing pipeline.

In [ ]:
import sqlite3
from ydata_profiling import ProfileReport
import pandas as pd
import numpy as np
import re
import os
import copy

## Run metadata and storage paths

Capture the execution timestamp and configure local folders / SQLite filenames used to persist intermediate outputs.

In [ ]:
today = pd.Timestamp("today").strftime("%Y%m%d_%H%M%S")
save_location = "/path/to/mepram_data"
db_filename = "db_mepram_sepsis_vf.sqlite3"

## Load database tables

Connect to the configured SQLite database and pull every `tbl_*` table into the `dataframes` dictionary for downstream transformations.

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import re
import os
con = sqlite3.connect(os.path.join(save_location, db_filename))

cursor = con.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

"""For older versions of the dataset
with open(os.path.join(save_location, "tbl_personid2center.sql"), "r") as f:
    sql_script = f.read()
    cursor.executescript(sql_script)

with open(os.path.join(save_location, "tbl_microorganismos.sql"), "r") as f:
    sql_script = f.read()
    cursor.executescript(sql_script)"""

tables = [row[0] for row in cursor.fetchall()]

print(tables)

dataframes = {}
for table in tables:
    print(f"Cargando la tabla: {table}")
    dataframes[table] = pd.read_sql_query(f"SELECT * FROM {table}", con)
con.close()

# Load tbl_codes2names for name mapping
tbl_codes2names = dataframes["tbl_codes2names"]

## Utility: handle outliers and categorise continuous variables

Define `process_variables` to flag IQR-based outliers, replace them with `NaN`, and bucket continuous values into ordinal ranges.

In [ ]:
def process_variables(df, variables):
    """
    This function replaces outliers with NaN using IQR and recodes quantitative variables into ranges 0-3.

    Parameters:
    df (pd.DataFrame): The DataFrame containing the data.
    variables (list): List of column names to process.

    Returns:
    pd.DataFrame: The DataFrame with outliers replaced by NaN and new recoded columns.
    """

    df_processed = df.copy()
    
    for var in variables:
        if var not in df.columns:
            print(f"Variable '{var}' not found in the DataFrame. It will be skipped.")
            continue

        Q1 = df_processed[var].quantile(0.25)
        Q3 = df_processed[var].quantile(0.75)
        IQR = Q3 - Q1

        lower_limit = Q1 - 3 * IQR
        upper_limit = Q3 + 3 * IQR

        outliers_mask = (df_processed[var] < lower_limit) | (df_processed[var] > upper_limit)
        outliers_count = outliers_mask.sum()
        if outliers_count > 0:
            print(f"{outliers_count} outliers replaced with NaN in the variable '{var}'.")
            print(f"{df_processed.loc[outliers_mask, var]}")            

        df_processed.loc[outliers_mask, var] = np.nan

        new_column = f"{var}_recoded"

        df_processed[new_column] = pd.qcut(df_processed[var], 
                                           q=4, 
                                           labels=[0, 1, 2, 3], 
                                           duplicates='drop')

        df_processed[new_column] = df_processed[new_column].astype('Int64') 
        
        print(f"Variable '{var}' recoded in the column '{new_column}'.")
    
    return df_processed

## Patient master record (`tbl_paciente`)

Combine base patient information with centre identifiers and derive flags such as previous infection and resistant organism history.

In [ ]:
df_pacientes = dataframes["tbl_paciente"].merge(dataframes["tbl_personid2center"], how="left")

df_pacientes["inf_previa_sino"] = np.where(np.isin(df_pacientes["person_id"].values, dataframes['tbl_infecciones_previas']["person_id"].values) == True, 1, 0)

bmr_previa = dataframes['tbl_infecciones_previas'].groupby("person_id")["bmr_infec_previa"].apply(lambda x: (x > 0).any())

df_pacientes = df_pacientes.merge(bmr_previa, on="person_id", how="left").fillna(False)
df_pacientes["bmr_infec_previa"] = np.where(df_pacientes["bmr_infec_previa"], 1, 0)


## Comorbidity features (`tbl_comorbilidad`)

One-hot encode cancer and hepatopathy categories to expose chronic-condition indicators for modelling.

In [ ]:
tbl_comorbilidad = dataframes['tbl_comorbilidad']
tbl_comorbilidad = pd.get_dummies(tbl_comorbilidad, columns=["tipo_cancer","tipo_hepatopatia"])
print(tbl_comorbilidad)

## BMR risk factor tracker (`tbl_factores_riesgo_bmr`)

Inspect multi-drug resistance risk factors and prepare them for merges without further transformation.

In [ ]:
tbl_factores_riesgo_bmr = dataframes['tbl_factores_riesgo_bmr']
print(tbl_factores_riesgo_bmr)

## Symptom presentation (`tbl_sintomas`)

- Pivot symptom duration into wide format to accumulate days per symptom.
- Derive presence/absence binaries and categorical duration buckets (0: none, 1: acute ≤7d, 2: prolonged >7d).

In [ ]:
sintom_map = tbl_codes2names[tbl_codes2names["variable"] == "sintoma"][["value", "name"]].to_dict(orient="records")
sintom_map = {str(float(x["value"])): x["name"].split(" | ")[-1] for x in sintom_map}

In [ ]:
tbl_sintomas = dataframes['tbl_sintomas'].copy()
tbl_sintomas["sintoma"] = tbl_sintomas["sintoma"].astype(str)
tbl_sintomas["sintoma"] = tbl_sintomas["sintoma"].map(sintom_map)
tbl_sintomas["sintoma"] = "sintoma_" + tbl_sintomas["sintoma"]
tbl_sintomas_pivoted = tbl_sintomas.pivot_table(
    index=["person_id", "fecha_ingreso_urgencias"],  
    columns="sintoma",
    values="duracion_sintoma",
    aggfunc="sum",
    fill_value=0).reset_index()

presence_absence= tbl_sintomas_pivoted.copy()
presence_absence.iloc[:, 2:] = (presence_absence.iloc[:, 2:] > 0).astype(int)

recategorized= tbl_sintomas_pivoted.copy()
recategorized.iloc[:, 2:] = recategorized.iloc[:, 2:].applymap(
    lambda x:0 if x== 0 else (1 if x <= 7 else 2)
)

tbl_sintomas_complete = presence_absence.merge(
    recategorized, on= ['person_id', 'fecha_ingreso_urgencias'], suffixes= ("", "_categorico")
)

print(tbl_sintomas_complete)

## Vital sign measurements (`tbl_signos`)

Parse numeric readings, normalise units, and derive clinical flags (hypotension, tachypnea, tachycardia, hypoxemia).

In [ ]:
tbl_signos = dataframes['tbl_signos']

for col in tbl_signos.columns[2:]:
    tbl_signos[col] = tbl_signos[col].apply(lambda x: re.findall(r'\d+\.\d+|\d+', str(x)))
    tbl_signos[col] = tbl_signos[col].apply(lambda x: float(x[0]) if x else None)
    
tbl_signos['hipotermia_hipertermia'] = np.where(tbl_signos['temperatura'] >= 38, 2,
                                np.where(tbl_signos['temperatura'] >= 36, 0, 1))
tbl_signos['hipotermia_hipertermia'] = tbl_signos['hipotermia_hipertermia'].where(tbl_signos['temperatura'].notna())

#tbl_signos = tbl_signos.drop(columns=['hipotermia_hipertermia'])
tbl_signos['hipotension'] = np.where(tbl_signos['tension_arterial'].isna(), tbl_signos['hipotension'],
                                     np.where(tbl_signos['tension_arterial'] <= 100, 1, 0))
tbl_signos['taquipnea'] = np.where(tbl_signos['frec_respiratoria'].isna(), tbl_signos['taquipnea'],
                                   np.where(tbl_signos['frec_respiratoria'] > 20, 1, 0))
tbl_signos['taquicardia'] = np.where(tbl_signos['frec_cardiaca'].isna(), tbl_signos['taquicardia'],
                                     np.where(tbl_signos['frec_cardiaca'] >90, 1, 0))
tbl_signos['hipoxemia'] = np.where(tbl_signos['saturacion_o2'].isna(), tbl_signos['hipoxemia'],
                                     np.where(tbl_signos['saturacion_o2'] >90, 0, 1))
print(tbl_signos)

## Apply IQR-based cleaning to vital sign variables

Use the helper to remove extreme values from key vital sign metrics before merging them into the main dataset.

In [ ]:
vairables= ['temperatura', 'frec_respiratoria', 'frec_cardiaca', 'tension_arterial','saturacion_o2']
tbl_signos = process_variables(tbl_signos, vairables)

## Sepsis assessment (`tbl_sepsis`)

Numerically encode sepsis questionnaire responses (e.g., lactate) and run the outlier-handling helper on selected labs.

In [ ]:
tbl_sepsis = dataframes['tbl_sepsis']
tbl_sepsis['lactato_serico'] = np.where(tbl_sepsis['lactato_serico'] == '<= 2 millimole per liter', 0, 1)
for col in tbl_sepsis.columns[2:]:
    tbl_sepsis[col] = tbl_sepsis[col].apply(lambda x: re.findall(r'\d+\.\d+|\d+', str(x)))
    tbl_sepsis[col] = tbl_sepsis[col].apply(lambda x: float(x[0]) if len(x) > 0 else None)
print(tbl_sepsis)

# Remove outliers and recode

variables =['proteina_c_reactiva']
tbl_sepsis = process_variables(tbl_sepsis, variables)

## Previous infection history (`tbl_infecciones_previas`)

- Map microorganism SNOMED codes to clinical categories and pivot them into dummy features.
- Count infection events per patient and compute inter-event timing statistics.

In [ ]:
# Load table with classification group for each organism in column label
organism_classification = dataframes["tbl_microorganismos"].copy()

# Rename default values for final classification, use _ to separate target variables
label_map = {
    "NOEB": "_Other bacteria",
    "VIRUS": "_Virus",
    "FUNGUS": "_Fungi",
    "OEB": "_Enterobacteria",
    "ECOLI": "Escherichia coli",
    "SA": "Staphylococcus aureus",
    "PSA": "Pseudomonas aeruginosa",
    "KP": "Klebsiella pneumoniae",
    "SP": "Streptococcus pneumoniae",
    "EC": "Enterococcus"
}
organism_classification["label"] = organism_classification["label"].map(label_map)
# organism_classification[organism_classification.duplicated(subset="snomed_code", keep=False)].sort_values(by=["snomed_code", "label"])
# There are snomed_codes with 2 labels, so always keep the most informative
organism_classification = organism_classification.sort_values(by=["snomed_code", "label"]).drop_duplicates(subset="snomed_code", keep="first")
# Create a dictionary with {organism-snomed-code (value) : classification group (label)}
organism_codes_map = dict(zip(organism_classification["snomed_code"], organism_classification["label"])) #label
print(organism_codes_map)

In [ ]:
tbl_infecciones_previas_mer = dataframes['tbl_infecciones_previas'].copy()


# Pasamos a dummy los microorganismos y rellenamos la información con el sumatorio de cada dummy. De este modo no perdemos información.
# Finalmente lo almacenamos en tbl_infecciones_previas_microorganismo
tbl_infecciones_previas_mer['grupo_microorganismo'] = tbl_infecciones_previas_mer['microorganism_infec_prev'].astype(str).map(organism_codes_map)
tbl_infecciones_previas_mer = tbl_infecciones_previas_mer.drop(columns=['microorganism_infec_prev', 'bmr_infec_previa', 'feno_resist_infec_prev'])
tbl_infecciones_previas_mer["dummy"] = 1
#tbl_infecciones_previas_mer = pd.get_dummies(tbl_infecciones_previas_mer, columns=['feno_resist_infec_prev'])
tbl_infecciones_previas_microorganismos = tbl_infecciones_previas_mer.pivot_table(
    index=["person_id", "fecha_ingreso_urgencias"],  
    columns="grupo_microorganismo",
    values="dummy",
    aggfunc="sum",
    fill_value=0).reset_index()

## Antibiotic treatments prior to admission (`tbl_tratamiento_antibiotico_previo`)

Filter to patients receiving pre-admission antimicrobials, aggregate treatment durations, and derive summary indicators.

In [ ]:
tbl_antib_prev = copy.deepcopy(dataframes["tbl_tratamiento_antibiotico_previo"])
tbl_antib_prev["antib_previo_si_no"] = np.where(tbl_antib_prev["dias_trat_antimicrobiano"] > 0, 1, 0)
tbl_antib_prev = tbl_antib_prev[tbl_antib_prev["antib_previo_si_no"] == 1]

### Map antibiotic codes to names in ultimo_antib column
antibmap = tbl_codes2names[tbl_codes2names["variable"] == "antimicrobiano_previo"][["value", "name"]].to_dict(orient="records")
antibmap = {x["value"]: x["name"].split(" | ")[-1].split("; ")[0] for x in antibmap}

tbl_antib_prev["antimicrobiano_previo"] = tbl_antib_prev["antimicrobiano_previo"].map(antibmap)
tbl_antib_prev["prev_betalactamase_inhib"] = (
    tbl_antib_prev["antimicrobiano_previo"]
    .str.contains("beta-lactamase inhibitor")
    .groupby(tbl_antib_prev["person_id"])   # adjust column name
    .transform("max")
    .astype(int)
)

### Group antimicrobial treatment based on clinical criteria

In [ ]:
antimicrobial_groups = [
    ("AMIKACINA", "Aminoglucósidos", "amikacin"),
    ("AMOXICILINA", "Penicilinas", "amoxicillin"),
    ("AMOXICILINA / CLAVULANICO", "Penicilinas", "amoxicillin and beta-lactamase inhibitor"),
    ("AMPICILINA", "Penicilinas", "ampicillin"),
    ("AZITROMICINA", "Macrólidos", "azithromycin"),
    ("AZTREONAM", "Monobactámicos", "aztreonam"),
    ("BENCILPENICILINA", "Penicilinas", None),
    ("BENCILPENICILINA-BENZATINA", "Penicilinas", None),
    ("CEFADROXILO MONOHIDRATO", "Cefalosporinas 1 gen", "cefadroxil"),
    ("CEFAZOLINA", "Cefalosporinas 1 gen", "cefazolin"),
    ("CEFEPIMA", "Cefalosporinas 4 gen", "cefepime"),
    ("CEFIDEROCOL", "Cefalosporinas 4 gen", None),
    ("CEFIXIMA", "Cefalosporinas 3 gen", "cefixime"),
    ("CEFOTAXIMA", "Cefalosporinas 3 gen", "cefotaxime"),
    ("CEFTAROLINA FOSAMILO", "Cefalosporinas 2 gen", "ceftaroline fosamil"),
    ("CEFTAZIDIMA", "Cefalosporinas 3 gen", "ceftazidime"),
    ("CEFTAZIDIMA / AVIBACTAM", "Cefalosporinas 3 gen", "ceftazidime and beta-lactamase inhibitor"),
    ("CEFTOLOZANO / TAZOBACTAM", "Cefalosporinas 3 gen", "ceftolozane and beta-lactamase inhibitor"),
    ("CEFTRIAXONA", "Cefalosporinas 3 gen", "ceftriaxone"),
    ("CEFUROXIMA", "Cefalosporinas 2 gen", "cefuroxime"),
    ("CIPROFLOXACINO", "Quinolonas", "ciprofloxacin"),
    ("CLARITROMICINA", "Macrólidos", "clarithromycin"),
    ("CLINDAMICINA", "Lincosamidas", "clindamycin"),
    ("CLOXACILINA", "Penicilinas", "cloxacillin"),
    ("COLISTIMETATO DE SODIO", "Polimixinas", None),
    ("DALBAVANCINA", "Glicopéptidos", "dalbavancin"),
    ("DAPTOMICINA", "Lipopéptidos", "daptomycin"),
    ("DOXICICLINA", "Tetraciclinas", "doxycycline"),
    ("ERITROMICINA", "Macrólidos", "erythromycin"),
    ("ERTAPENEM", "Carbapenemas", "ertapenem"),
    ("FIDAXOMICINA", "Macrólidos", None),
    ("FOSFOMICINA", "Fosfomicina", "fosfomycin"),
    ("FOSFOMICINA-TROMETAMOL", "Fosfomicina", "fosfomycin"),
    ("GENTAMICINA", "Aminoglucósidos", "gentamicin"),
    ("IMIPENEM", "Carbapenemas", "imipenem and cilastatin"),
    ("IMIPENEM/RELEBACTAM", "Carbapenemas", None),
    ("LEVOFLOXACINO", "Quinolonas", "levofloxacin"),
    ("LINEZOLID", "Lincosamidas", "linezolid"),
    ("MEROPENEM", "Carbapenemas", "meropenem"),
    ("METRONIDAZOL", "Metronidazol", "metronidazole"),
    ("MINOCICLINA", "Minociclina", None),
    ("MOXIFLOXACINO", "Quinolonas", "moxifloxacin"),
    ("NITROFURANTOINA", "Nitrofurantoína", "nitrofurantoin"),
    ("NORFLOXACINO", "Quinolonas", "norfloxacin"),
    ("PIPERACILINA / TAZOBACTAM", "Penicilinas", "piperacillin and beta-lactamase inhibitor"),
    ("POSACONAZOL", "Azoles", "posaconazole"),
    ("SULFADIAZINA", "Sulfonamidas", "sulfadiazine"),
    ("SULFAMETOXAZOL / TRIMETOPRIMA", "Sulfonamidas", "sulfamethoxazole and trimethoprim"),
    ("TRIMETOPRIMA", "Sulfonamidas", "trimethoprim"),
    ("TEICOPLANINA", "Glicopéptidos", "teicoplanin"),
    ("TIGECICLINA", "Tigeciclina", "tigecycline"),
    ("TOBRAMICINA", "Aminoglucósidos", "tobramycin"),
    ("VANCOMICINA", "Glicopéptidos", "vancomycin"),
    ("A", "Tetraciclinas", "tetracycline"),
    ("A", "Cefalosporinas 3 gen", "cefditoren"),
    ("A", "Carbapenemasas", "meropenem and vaborbactam"),
    ("A", "Cefalosporinas 1 gen", "cefalexin"),
    ("A", "Cefalosporinas 2 gen", "cefoxitin"),
    ("A", "AntiTuberculoso", "rifampicin"),
    ("A", "AntiTuberculoso","rifabutin"),
    ("A", "AntiTuberculoso","isoniazid"),
    ("A", "AntiTuberculoso","pyrazinamide"),
    ("A", "AntiTuberculoso","ethambutol"),
    ("B", "Azoles", "isavuconazole"),
    ("B", "Azoles", "fluconazole"),
    ("B", "Azoles", "itraconazole"),
    ("B", "Azoles", "voriconazole"),
    ("B", "Equinocandinas", "caspofungin"),
    ("B", "Equinocandinas", "micafungin"),
    ("B", "Equinocandinas", "anidulafungin"),
]
antimicrobial_groups = {x[2]: x[1] for x in antimicrobial_groups}
tbl_antib_prev["antimicrobiano_previo"] = tbl_antib_prev["antimicrobiano_previo"].map(antimicrobial_groups)
tbl_antib_prev["antimicrobiano_previo"].value_counts()

In [ ]:
tbl_antib_prev["antimicrobiano_previo"].value_counts()

In [ ]:
tbl_antib_prev["ultimo_antib"] = (
    tbl_antib_prev.loc[tbl_antib_prev.groupby("person_id")["fecha_ingreso_urgencias"].idxmax(), ["person_id", "antimicrobiano_previo"]]
      .set_index("person_id")["antimicrobiano_previo"]
      .reindex(tbl_antib_prev["person_id"])
      .values
)
antib_prev_pivoted = tbl_antib_prev.drop(columns="via_administ_antib_prev").pivot_table(
    index=["person_id", "fecha_ingreso_urgencias", "antib_previo_si_no", "ultimo_antib", "prev_betalactamase_inhib"],
    columns="antimicrobiano_previo",
    values="dias_trat_antimicrobiano",
    aggfunc="sum",
    fill_value=0
).reset_index()
antib_prev_pivoted["antib_previo_total_dias"] = antib_prev_pivoted.iloc[:, 5:].sum(axis=1)
antib_prev_to_merge = antib_prev_pivoted.copy()

## Hemoculture results from the emergency department

Deep-copy the hemoculture table, map microorganism codes to organism groups, and inject clinician-reviewed co-infection resolutions for the target label.

In [ ]:
import copy
hemo_urg = copy.deepcopy(dataframes['tbl_hemocultivo_de_urgencias'])
hemo_urg["microorganismo"] = hemo_urg["microorganismo"].fillna("0").astype(int).astype(str).map(lambda x: organism_codes_map.get(x, "NEGATIVE"))
hemo_urg["microorganismo"].value_counts()

In [ ]:
# Resolución de co-infecciones (manual-clinico)
coinf_res_dict = {
    50: "Escherichia coli",
    419: "Staphylococcus aureus",
    770: "Escherichia coli",
    780: "Escherichia coli",
    1214: "Klebsiella pneumoniae",
    1309: "Escherichia coli",
    1886: "Klebsiella pneumoniae",
    1925: "Escherichia coli",
    1960: "Enterococcus avium",
    1977: "Escherichia coli",
    2274: "Escherichia coli",
    2294: "Contaminación",
    2634: "Escherichia coli",
    2638: "Klebsiella pneumoniae",
    2681: "Klebsiella pneumoniae",
    2845: "Escherichia coli",
    2971: "Escherichia coli",
    2983: "Escherichia coli",
    3019: "Escherichia coli",
    3087: "Escherichia coli",
    3167: "Escherichia coli",
    3352: "Escherichia coli",
    3479: "Escherichia coli",
    3819: "Otra enterobacteria",
    1247: "Otra enterobacteria",
    2640: "Escherichia coli",
    133: "Pseudomonas aeruginosa",
    1881: "Escherichia coli",
    420: "Pseudomonas fluorescens",
    1110: "Clostridium perfringens",
    1239: "Klebsiella pneumoniae",
    1310: "Escherichia coli",
    1379: "Pseudomonas aeruginosa",
    1906: "Candida glabrata",
    2041: "Escherichia coli",
    2284: "Fusobacterium nucleatum",
    2647: "Otra enterobacteria",
    2848: "Escherichia coli",
    3116: "Escherichia coli",
    447: "Escherichia coli",
    607: "Otra enterobacteria",
    668: "Klebsiella pneumoniae",
    760: "Escherichia coli",
    956: "Streptococcus anginosus",
    1089: "Staphylococcus aureus",
    1170: "Escherichia coli",
    1228: "Escherichia coli",
    1286: "Escherichia coli",
    1308: "Klebsiella pneumoniae",
    1311: "Candida glabrata",
    1322: "Otra enterobacteria",
    1355: "Escherichia coli",
    1663: "Escherichia coli",
    1717: "Contaminación",
    1865: "Bacteroides fragilis",
    1885: "Otra enterobacteria",
    1893: "Pseudomonas aeruginosa",
    1967: "Escherichia coli",
    1975: "Escherichia coli",
    1976: "Otra enterobacteria",
    2336: "Escherichia coli",
    2441: "Escherichia coli",
    2469: "Escherichia coli",
    2926: "Klebsiella pneumoniae",
    2974: "Klebsiella pneumoniae",
    3457: "Otra enterobacteria",
    1954: "Escherichia coli",
    2016: "Escherichia coli",
    2810: "Escherichia coli",
    2840: "Staphylococcus aureus",
    429: "Streptococcus pyogenes",
    473: "Escherichia coli",
    480: "Staphylococcus aureus",
    766: "Contaminación",
    797: "Otra enterobacteria",
    1303: "Contaminación",
    1777: "Salmonella enterica",
    1839: "Escherichia coli",
    1850: "Escherichia coli",
    1873: "Otra enterobacteria",
    2308: "Contaminación",
    2335: "Otra enterobacteria",
    2341: "Staphylococcus aureus",
    2513: "Otra enterobacteria",
    2873: "Klebsiella pneumoniae",
    462: "Escherichia coli",
    562: "Escherichia coli",
    599: "Escherichia coli",
    759: "Otra enterobacteria",
    871: "Pseudomonas aeruginosa",
    961: "Escherichia coli",
    1052: "Otra enterobacteria",
    1241: "Escherichia coli",
    1246: "Klebsiella pneumoniae",
    1333: "Escherichia coli",
    1658: "Escherichia coli",
    1776: "Escherichia coli",
    1875: "Escherichia coli",
    1956: "Otra enterobacteria",
    1966: "Escherichia coli",
    2017: "Escherichia coli",
    2303: "Staphylococcus aureus",
    2321: "Escherichia coli",
    2354: "Klebsiella pneumoniae",
    2408: "Escherichia coli",
    2467: "Escherichia coli",
    2479: "Klebsiella pneumoniae",
    2493: "Pseudomonas aeruginosa",
    2533: "Escherichia coli",
    2625: "Escherichia coli",
    2663: "Escherichia coli",
    2715: "Escherichia coli",
    2813: "Klebsiella pneumoniae",
    3491: "Otra enterobacteria",
    3675: "Klebsiella pneumoniae",
    426: "Staphylococcus aureus"
}
coinf_mapping = {
    "Candida glabrata": "_Fungi",
    "Clostridium perfringens": "_Other bacteria",
    "Contaminación": "NEGATIVE",
    "Enterococcus avium": "Enterococcus",
    "Escherichia coli": "Escherichia coli",
    "Fusobacterium nucleatum": "_Other bacteria",
    "Klebsiella pneumoniae": "Klebsiella pneumoniae",
    "Otra enterobacteria": "_Enterobacteria",
    "Pseudomonas aeruginosa": "Pseudomonas aeruginosa",
    "Pseudomonas fluorescens": "_Other bacteria",
    "Salmonella enterica": "_Enterobacteria",
    "Staphylococcus aureus": "Staphylococcus aureus",
    "Streptococcus anginosus": "_Other bacteria",
    "Streptococcus pyogenes": "_Other bacteria",
    "Bacteroides fragilis": "_Other bacteria",
}
coinf_res_dict = {k: coinf_mapping[v] for k,v in coinf_res_dict.items()}
hemo_urg["microorganismo"] = hemo_urg["person_id"].map(coinf_res_dict).fillna(hemo_urg["microorganismo"])

### Pivot hemoculture findings to patient-level features

Deduplicate hemoculture records, pivot microorganism flags into a wide format, and compute per-visit outcome tuples such as `resultado_hemo`.

In [ ]:
hemo_urg_grouped = (
    hemo_urg.groupby(["person_id", "microorganismo"], as_index=False)
      .agg({
          "fecha_ingreso_urgencias": "first",
          "id_hemocultivo": "first",
          "fecha_hemocultivo": "first",
          "hemo_positivo_si_no": "first",
          "bmr_etiologia": "max",
          "fenotipo_resistencia": lambda x: tuple(0.0 if pd.isna(v) else v for v in x)
      })
)
hemo_urg_grouped = hemo_urg_grouped.fillna({"bmr_etiologia": 0.0})
hemo_urg_pivoted = hemo_urg_grouped.pivot_table(
    index=["person_id", "fecha_ingreso_urgencias", "bmr_etiologia", "fenotipo_resistencia"],
    columns="microorganismo",
    values="hemo_positivo_si_no",
    aggfunc="sum",
    fill_value=0).reset_index()
hemo_urg_pivoted

In [ ]:

def get_bacteria(row):
    return ", ".join([col for col in row.index if row[col] == 1])

hemo_urg_pivoted["resultado_hemo"] = hemo_urg_pivoted.drop(columns=["person_id", "fecha_ingreso_urgencias", "bmr_etiologia", "fenotipo_resistencia"]).apply(get_bacteria, axis=1)
hemo_urg_pivoted["resultado_hemo"] = hemo_urg_pivoted["resultado_hemo"].replace("", "NEGATIVE").apply(lambda x: tuple(x.split(", ")))

## Colonisation history (`tbl_colonizaciones_previas`)

Map colonising organisms to grouped labels and pivot them into patient-level indicator features.

In [ ]:
colo_prev = copy.deepcopy(dataframes["tbl_colonizaciones_previas"])
colo_prev["microorganism_colonizador"] = colo_prev["microorganism_colonizador"].astype(str).map(organism_codes_map)

In [ ]:
colo_prev["dummy"] = 1
colo_prev_pivoted = colo_prev.pivot_table(
    index=["person_id", "fecha_ingreso_urgencias"],  
    columns="microorganism_colonizador",
    values="dummy",
    aggfunc="sum",
    fill_value=0).reset_index()
colo_prev_pivoted = colo_prev_pivoted.drop(columns=["fecha_ingreso_urgencias"])
colo_prev_pivoted.columns = ["colo_" + str(col) if col not in ["person_id", "fecha_ingreso_urgencias"] else col for col in colo_prev_pivoted.columns]

In [ ]:
# Calculo del número de eventos por paciente y el tiempo medio entre cada visita. 

tbl_visitas_ip = tbl_infecciones_previas_mer.copy()
fechas_invalidas = tbl_visitas_ip[~tbl_visitas_ip['fecha_infeccion'].str.match(r'\d{4}-\d{2}-\d{2}')]
tbl_visitas_ip.loc[tbl_visitas_ip['fecha_infeccion'] == '323-05-01', 'fecha_infeccion'] = '2023-05-01'
tbl_visitas_ip['fecha_infeccion'] = pd.to_datetime(tbl_visitas_ip['fecha_infeccion'])
tbl_visitas_ip = tbl_visitas_ip[['person_id', 'fecha_ingreso_urgencias', 'fecha_infeccion']]
num_visitas = tbl_visitas_ip.groupby('person_id')['fecha_infeccion'].nunique().reset_index(name='num_inf_previas')
print(num_visitas)

## Time since last infection at admission

Compute visit-level timelines: number of infections per patient, days since the last event, and average gaps for recurrent cases.

In [ ]:
# Calculo de dias desde desde la última infección hasta el ingreso

tbl_visitas_ip['fecha_ingreso_urgencias'] = pd.to_datetime(tbl_visitas_ip['fecha_ingreso_urgencias'])
ultima_infeccion = tbl_visitas_ip.groupby('person_id')['fecha_infeccion'].max().reset_index(name= 'ultima_fecha')
ultima_infeccion['ultima_fecha'] = pd.to_datetime(ultima_infeccion['ultima_fecha'])
fecha_ingreso_unica = tbl_visitas_ip[['person_id', 'fecha_ingreso_urgencias']].drop_duplicates()
ultima_infeccion_df = ultima_infeccion.merge(fecha_ingreso_unica, on='person_id')
ultima_infeccion_df['tiempo_ultima'] = (ultima_infeccion_df['fecha_ingreso_urgencias'] - ultima_infeccion_df['ultima_fecha']).dt.days

In [ ]:
# merge infecciones previas

tbl_infecciones_complete = tbl_infecciones_previas_microorganismos.merge(num_visitas, on= ['person_id'], how= 'left')
tbl_infecciones_complete = tbl_infecciones_complete.merge(ultima_infeccion_df, on= ['person_id'], how= 'left')
tbl_infecciones_complete = tbl_infecciones_complete.drop(columns=['fecha_ingreso_urgencias_y'])
tbl_infecciones_complete.head(5)

## Other emergency cultures (no resistance data)

Standardise organism codes, pivot additional culture indicators, and prepare them for merging with hemoculture outcomes.

In [ ]:
tbl_otros_cultivos_en_urgencias = copy.deepcopy(dataframes['tbl_otros_cultivos_en_urgencias'])
tbl_otros_cultivos_en_urgencias["otro_cult_microorganismo"] = tbl_otros_cultivos_en_urgencias["microorganismo_otros_cult"].fillna("0").astype(int).astype(str).map(lambda x: organism_codes_map.get(x, "NEGATIVE"))
tbl_otros_cultivos_en_urgencias = tbl_otros_cultivos_en_urgencias.drop(columns=["microorganismo_otros_cult", "id_otros_cultivos", "fecha_otros_cultivos", "bmr_etiologia_otros", "fenotipo_resistencia_otros"])
tbl_otros_cultivos_en_urgencias["tipo_cultivo"] = tbl_otros_cultivos_en_urgencias["tipo_cultivo"].fillna(0)
tbl_otros_cultivos_en_urgencias = tbl_otros_cultivos_en_urgencias.dropna().drop_duplicates()
tbl_otros_cultivos_en_urgencias["dummy"] = 1
tbl_otros_cultivos_en_urgencias_pivoted = tbl_otros_cultivos_en_urgencias.pivot_table(
    index=["person_id", "fecha_ingreso_urgencias"],  
    columns="otro_cult_microorganismo",
    values="dummy",
    aggfunc="sum",
    fill_value=0).reset_index()
tbl_otros_cultivos_en_urgencias_pivoted = tbl_otros_cultivos_en_urgencias_pivoted.rename(columns=lambda x: f"otros_cult_{x}" if x not in ["person_id", "fecha_ingreso_urgencias"] else x)
tbl_otros_cultivos_en_urgencias_pivoted

In [ ]:
def pick_dominant(row):
    if (row >= 2).any():
        return row.idxmax()  # the one with value 2
    elif (row == 1).sum() == 1:
        return row[row == 1].index[0]  # the first one present once (probably bacteria + negative)
    else:
        return "NEGATIVE"

org_list = list(set(organism_codes_map.values()))
print(org_list)
all_urg = tbl_otros_cultivos_en_urgencias_pivoted.merge(hemo_urg_pivoted, on=["person_id", "fecha_ingreso_urgencias"], how="outer")
all_urg_comb = (
    all_urg.rename(columns=lambda c: c.replace('otros_cult_', ''))
      .groupby(axis=1, level=0)
      .sum()
)
all_urg_comb = all_urg_comb[[c for c in all_urg_comb.columns if c != "NEGATIVE"] + ["NEGATIVE"]]
all_urg_comb["all_cult_org"] = all_urg_comb[org_list].apply(pick_dominant, axis=1)
all_urg_comb

In [ ]:
foco_map = tbl_codes2names[tbl_codes2names["variable"].str.contains("foco")][["value", "name"]].apply(lambda x: x.str.replace("foco | ", ""), axis=1)
foco_map = {int(x): v for x,v in foco_map.set_index("value")["name"].to_dict().items()}

### Extract co-infection patients

In [ ]:
"""co_inf_patients = pd.merge(tbl_sepsis, hemo_urg_pivoted)[["person_id", "foco", "resultado_hemo"]]
co_inf_patients = co_inf_patients[co_inf_patients["resultado_hemo"].apply(lambda x: len(x) > 1)]
co_inf_patients["foco"] = co_inf_patients["foco"].map(foco_map)
co_inf_patients["resultado_hemo"] = co_inf_patients["resultado_hemo"].apply(lambda x: ", ".join(x)).str.replace(" (organismo)", "")
co_inf_patients.to_excel(os.path.join(save_location, "co_infection_patients.xlsx"), index=False)"""

## Assemble the modelling dataset

Merge patient characteristics, comorbidities, risk factors, cultures, and outcomes into a single training-ready dataframe.

In [ ]:
df_merged = df_pacientes.merge(tbl_comorbilidad, on = ['person_id', 'fecha_ingreso_urgencias'], how= 'left')
df_merged = df_merged.merge(tbl_factores_riesgo_bmr, on = ['person_id', 'fecha_ingreso_urgencias'], how= 'left')
df_merged = df_merged.merge(tbl_sepsis, on= ['person_id', 'fecha_ingreso_urgencias'], how= 'left')
df_merged = df_merged.merge(tbl_signos, on= ['person_id', 'fecha_ingreso_urgencias'], how= 'left')
df_merged = df_merged.merge(tbl_sintomas_complete, on= ['person_id', 'fecha_ingreso_urgencias'], how= 'left')
orig_cols = df_merged.columns.tolist()
df_merged = df_merged.merge(tbl_infecciones_complete, on= ['person_id'], how= 'left')
df_merged = df_merged.merge(colo_prev_pivoted, on = ['person_id'], how= 'left')
df_merged = df_merged.merge(antib_prev_to_merge, on= ['person_id', 'fecha_ingreso_urgencias'], how='left')
new_cols = [c for c in df_merged.columns if c not in orig_cols]
df_merged[new_cols] = df_merged[new_cols].fillna(0)
df_merged = pd.merge(df_merged, hemo_urg_pivoted[["person_id", "resultado_hemo", "bmr_etiologia", "fenotipo_resistencia"]], on= ['person_id'], how= 'left')
df_merged = pd.merge(df_merged, all_urg_comb[["person_id", "all_cult_org"]], on= ['person_id'], how= 'left')

In [ ]:
# Create a column that sums all microorganism events
for org in set(organism_codes_map.values()):
    cols = [col for col in df_merged.columns if org in col]
    # Option A: if you want sum of counts (if columns are counts)
    df_merged[f"{org}_total"] = df_merged[cols].sum(axis=1)

In [ ]:
# Merge all cancer and hepatic deseases
def merge_columns(df_to_clean, column_list, new_col_name):
    df_to_clean[new_col_name] = df_to_clean[column_list].sum(axis=1).astype(int)
    clean_df = df_to_clean.drop(columns=column_list)
    return clean_df

hepatic_cols = [c for c in df_merged.columns if "hepatopatia" in c]
tumor_cols = [c for c in df_merged.columns if "cancer" in c]
for new_name, col_list in {"hepatopatias_totales": hepatic_cols, "canceres_totales": tumor_cols}.items():
    try:
        df_merged = merge_columns(df_merged, col_list, new_name)
    except Exception as e:
        print(f"Error merging columns {col_list} into {new_name}: {e}")
df_merged["canceres_si_no"] = np.where(df_merged["canceres_totales"] > 0, 1, 0)
df_merged["hepatopatias_si_no"] = np.where(df_merged["hepatopatias_totales"] > 0, 1, 0)


In [ ]:
df_expanded = df_merged.explode("resultado_hemo").reset_index(drop=True)
df_expanded["foco"] = df_expanded["foco"].map(foco_map)
df_expanded["bmr_etiologia"] = np.where(df_expanded["bmr_etiologia"] == 1.0, "BMR resistente", "NEGATIVE")

# Create BMR + phenotype prediction dataframe

### Check resist phenotype distribution

In [ ]:
phenomap = {float(x["value"]):x["name"] for x in tbl_codes2names[tbl_codes2names["variable"] == "fenotipo_resistencia"][["value", "name"]].to_dict(orient="records")}

df_expanded["fenotipo_resistencia"].apply(
    lambda x: tuple(phenomap.get(n, "NEGATIVE").split(" resistente a ")[-1] for n in x)
).value_counts()

In [ ]:
df_resist = df_expanded.explode("fenotipo_resistencia").reset_index(drop=True)
person_freq = df_resist["person_id"].value_counts()
df_resist["sample_weight"] = df_resist["person_id"].map(1 / person_freq)

In [ ]:
df_resist["fenotipo_resistencia"] = df_resist["fenotipo_resistencia"].map(phenomap)

df_resist["fenotipo_resistencia"] = df_resist["fenotipo_resistencia"].apply(lambda x: x.split(" resistente a ")[-1] if not pd.isna(x) else x)

In [ ]:
df_resist["fenotipo_resistencia"].value_counts()

In [ ]:
antibdict = {
    "amoxicilina/clavulánico":     "amoxicillin and beta-lactamase inhibitor",     
    "ciprofloxacino":     "ciprofloxacin",                                
    "ceftrixona o cefotaxima":     "ceftriaxone",                                  
    "ceftazidima":     "ceftazidime",                                  
    "piperacilina/tazobactam":     "piperacillin and beta-lactamase inhibitor",    
    "cefepima":     "cefepime",                                     
    "ampicilina o penicilina":     "ampicillin",                                   
    "meticilina":     "cloxacillin",                                  
    "meropenem":     "meropenem",                                    
    "ceftazidima/avibactam":     "ceftazidime and beta-lactamase inhibitor",     
    "ceftolozano/tazobactam":     "ceftolozane and beta-lactamase inhibitor",     
    "vancomicina":     "vancomycin"                                    
}
#[antimicrobial_groups.get(x, x) for x in antibdict.values()]
fenotipo_groups = {x: antimicrobial_groups.get(v, v) for x,v in antibdict.items()}
df_resist["fenotipo_resistencia"] = df_resist["fenotipo_resistencia"].map(fenotipo_groups).fillna("NEGATIVE")
#df_resist["fenotipo_resistencia"] = df_resist["fenotipo_resistencia"].map(fenotipo_groups)

In [ ]:
dropped_for_na = []
for col in df_resist:
    na_per = df_resist[col].isna().mean()
    if na_per > 0.052:
        dropped_for_na.append(col)
print(df_resist["bmr_etiologia"].value_counts())
print(df_resist.drop(columns=dropped_for_na).dropna()["bmr_etiologia"].value_counts())

In [ ]:
#df_resist["sample_weight"] = 1
df_resist.to_csv(os.path.join(save_location, "df_resist_bmr_grouped.csv"), index=False)

In [ ]:
testdf = df_expanded[~df_expanded["resultado_hemo"].isin(["_Fungi",	"_Other bacteria", "Enterococcus"])]
testdf = testdf[~testdf["foco"].isin(["piel", "catéter venoso"
    ,"vías altas respiratorias"
    ,"cardiovascular"
    ,"osteoarticular"
    ,"sistema nervioso central"
    ,"genital",])]
freq_foco_resultado = (
    testdf.groupby(["foco", "resultado_hemo"], observed=True)
    .size()
    .unstack(fill_value=0)             # columns = foco, rows = resultado_hemo
)
freq_foco_resultado_pct = freq_foco_resultado.div(freq_foco_resultado.sum(axis=0), axis=1)*100
freq_foco_resultado_pct.applymap(lambda v: f"{v:.1f}%")

### Create "infected_yes_no" as a binary head for fenotype prediction

In [ ]:
df_expanded["infected_yes_no"] = np.where(df_expanded["resultado_hemo"] == "NEGATIVE", "NEGATIVE", "POSITIVE")

### Multi-hot encode fenotipo_resistencia for multi-label prediction

In [ ]:
df_expanded["fenotipo_resistencia"] = df_expanded["fenotipo_resistencia"].apply(
    lambda x: tuple(phenomap[n].split(" resistente a ")[-1] for n in x if n in phenomap)
)

### Group by pharmacological family

In [ ]:
df_expanded["fenotipo_resistencia"] = df_expanded["fenotipo_resistencia"].apply(
    lambda x: tuple(fenotipo_groups[n] for n in x if n in fenotipo_groups)
)

### Persist intermediate datasets

Write exploded datasets to disk so later steps can reload them without recomputing heavy joins.

In [ ]:
#df_expanded["foco"].unique()df_expanded["foco"]
#print(df_expanded["foco"].map(foco_map).value_counts())
print(df_expanded["resultado_hemo"].value_counts())
print("no NA:", df_expanded[~df_expanded["foco"].isna()].shape)
print("normal shape:", df_expanded.shape)
print("excluding minor focos: ", df_expanded[~df_expanded["foco"].map(foco_map).isin(["catéter venoso", "vías altas respiratorias", "cardiovascular", "osteoarticular", "sistema nervioso central", "genital"])]["resultado_hemo"].value_counts())

# Save the finished table

In [ ]:
def dedupe_labels(labels):
    return sorted(set(labels))

df_expanded["fenotipo_resistencia"] = df_expanded["fenotipo_resistencia"].apply(dedupe_labels)

In [ ]:
df_expanded["sample_weight"] = 1
df_expanded.to_csv(os.path.join(save_location, "df_merged_full_multilabel_grouped.csv"), index=False)

# Process bacthecom data

In [ ]:
import msoffcrypto
import io
import pandas as pd

password = "TESTPASSWORD"

with open("/path/to/bacthecom_hc_urgencias_v2_completo.xlsx", "rb") as file:
    office_file = msoffcrypto.OfficeFile(file)
    office_file.load_key(password=password)

    decrypted = io.BytesIO()
    office_file.decrypt(decrypted)

bacthecom_df = pd.read_excel(decrypted)
shared_cols = [col for col in bacthecom_df.columns if col in df_resist.columns]
bact_df_shared = bacthecom_df.copy()[shared_cols]


In [ ]:
df_expanded[~df_expanded["resultado_hemo"].isin([
    "Enterococcus",
    "_Fungi",
    "_Other bacteria",
])]["resultado_hemo"].value_counts()

# Optional: Data analysis

### Plot results from hierarchical_model_train_rfecv.py execution in hpc

In [ ]:
foco_map = {1.0: 'pulmonar',
 2.0: 'intraabdominal',
 3.0: 'biliar',
 4.0: 'urinario',
 5.0: 'cardiovascular',
 6.0: 'piel',
 7.0: 'sistema nervioso central',
 8.0: 'catéter venoso',
 9.0: 'vías altas respiratorias',
 10.0: 'osteoarticular',
 11.0: 'genital',
 12.0: 'desconocido'}

## Visualisation setup

Load plotting libraries and helper metadata that support the upcoming exploratory analyses.

In [ ]:
sintom_dict = tbl_codes2names[tbl_codes2names["name"].str.contains("síntoma ")][["value", "name"]].to_dict(orient="records")
sintom_dict = {"sintoma_"+str(float(d["value"])): "sintoma_" + d["name"].replace("síntoma | ", "") for d in sintom_dict}
sintom_map = {}
for k,v in sintom_dict.items():
    sintom_map[k] = v
    sintom_map[k+"_categorico"] = v+"_categorico"
sintom_map

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import LabelEncoder
import copy

def correlation_ratio(categories, measurements):
    """Correlation ratio (eta squared) between categorical target and numerical feature"""
    categories = np.array(categories)
    measurements = np.array(measurements)
    cat_groups = [measurements[categories == cat] for cat in np.unique(categories)]
    means = [np.mean(g) for g in cat_groups if len(g) > 0]
    n = len(measurements)
    grand_mean = np.mean(measurements)
    ss_between = sum(len(g) * (m - grand_mean) ** 2 for g, m in zip(cat_groups, means))
    ss_total = sum((measurements - grand_mean) ** 2)
    return np.sqrt(ss_between / ss_total) if ss_total > 0 else 0

# -------------------------------------
target = "resultado_hemo"
df_to_plot = df_expanded.copy().rename(columns=sintom_map)
df = copy.deepcopy(df_to_plot.drop(["freq_bac_foco", "freq_bacteria", "infected_yes_no", "person_id"], axis=1))
df = df[~df["resultado_hemo"].isin(["Enterococcus", "_Fungi", "_Other bacteria", '_Virus'])]

antibmap = {x["value"]:x["name"].split("|")[1].strip() for x in tbl_codes2names[tbl_codes2names["variable"] == "antimicrobiano_previo"][["value", "name"]].to_dict(orient="records")}
df["ultimo_antib"] = df["ultimo_antib"].map(antibmap)
df["ultimo_antib"] = LabelEncoder().fit_transform(df["ultimo_antib"])
#df = pd.get_dummies(df, columns=["ultimo_antib"], drop_first=True)
# Get categories directly
classes = df[target].unique()
numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != target]
"""df = pd.get_dummies(df, columns=["ultimo_antib"], drop_first=True)
df = df.rename(columns=lambda c: c.replace('ultimo_antib_', ''))
numeric_cols = df.drop(target, axis=1).columns"""
# Prepare a dataframe to store correlations per class and feature
corr_matrix = pd.DataFrame(index=classes, columns=numeric_cols, dtype=float)

for cls in classes:
    # binary target: this class vs the rest
    y_binary = (df[target] == cls).astype(int)
    for col in numeric_cols:
        corr_matrix.loc[cls, col] = correlation_ratio(y_binary, df[col].values)

# Optional: filter weak correlations
filtered = corr_matrix.loc[:, corr_matrix.max(axis=0) > 0.1]

# -------------------------------------
# 🔹 Plot heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(filtered, annot=False, cmap="viridis", cbar_kws={"label": "η (class-wise correlation)"})
plt.title(f"Ultimo_antib correlation ratio (η) by etiology. Showing only those with η > 0.02")
plt.xlabel("Feature")
plt.ylabel("Bacteria / Etiology class")
plt.tight_layout()
plt.show()


## Subset for etiology modelling

Create a reduced dataframe focused on pre-admission events (infections, colonisations) paired with hemoculture outcomes.

In [ ]:
df_hemo_merged = pd.merge(df_pacientes, hemo_urg_pivoted[["person_id", "resultado_hemo"]], on= ['person_id'], how= 'left')

orig_cols = df_hemo_merged.columns.tolist()
df_hemo_merged = df_hemo_merged.merge(tbl_infecciones_complete, on= ['person_id'], how= 'left')
df_hemo_merged = df_hemo_merged.merge(colo_prev_pivoted, on = ['person_id'], how= 'left')
new_cols = [c for c in df_hemo_merged.columns if c not in orig_cols]
df_hemo_merged[new_cols] = df_hemo_merged[new_cols].fillna(0) # Not all patients have previous infections/colonizations

df_hemo_merged = df_hemo_merged.merge(tbl_sepsis, on= ['person_id', 'fecha_ingreso_urgencias'], how= 'left')

In [ ]:
df_expanded = df_hemo_merged.explode("resultado_hemo").reset_index(drop=True)
counts = df_expanded["person_id"].value_counts()
df_expanded["weight"] = df_expanded["person_id"].apply(lambda x: 1 / counts[x])
df_expanded["co_infection"] = np.where(df_expanded["weight"] < 1, 1, 0)

In [ ]:
df_expanded.to_csv(os.path.join(save_location, "df_hemo_merged_v2.csv"), index=False)

## Generate automated EDA report

Reload the merged dataset and render an HTML profiling report with `ydata_profiling`.

In [ ]:
df_merged = pd.read_csv(os.path.join(save_location, "df_merged_full.csv"))

In [ ]:
profile = ProfileReport(df_merged, title="MePRAM EDA report")
profile.to_notebook_iframe()
profile.to_file(os.path.join(save_location, "df_merge_report.html"))

## Visualise missingness patterns

Plot heatmaps that highlight columns with substantial fractions of missing data to guide imputation strategies.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def insert_linebreak(string, lengLabel=10):
    return '\n'.join(string[i:i+lengLabel] for i in range(0, len(string), lengLabel))
merged_df = pd.read_csv(os.path.join(save_location, "df_merged.csv"))
prev_num = 0

def plot_missing_columns(subset_df, title=f"Missing Data Matrix for columns 0 to 171"):
    missing_df = subset_df.isna()

    missing_df.columns = subset_df.columns
    missing_percent = missing_df.mean() * 100
    columns_with_percent = [
        f"$\\bf{{{col}}}$ ({missing_percent[col]:.2f}%)" if missing_percent[col] > 30 else f"{col} ({missing_percent[col]:.2f}%)"
        for col in subset_df.columns
    ]
    plt.figure(figsize=(24, 6))
    ax = sns.heatmap(missing_df, vmin=0, vmax=1, cbar=False,
                xticklabels=columns_with_percent)
    ax.tick_params(axis='x', which='minor', length=40)
    plt.title(title)
    plt.xlabel("Columns (% Missing)")
    plt.ylabel("Rows", rotation=90)
    plt.xticks(rotation=90)
    plt.show()


plot_missing_columns(merged_df)

missing_df = merged_df.isna()
missing_df.columns = merged_df.columns
missing_percent = missing_df.mean() * 100
print([idx for idx,x in enumerate(missing_percent) if x > 30])
dangerous_df = merged_df.iloc[:, [idx for idx,x in enumerate(missing_percent) if x > 30]]
print(dangerous_df)
plot_missing_columns(dangerous_df, f"Missing Data Matrix of {len(dangerous_df.columns)} columns with > 30% NAs")

## Principal component analysis for feature exploration

Scale features with `MinMaxScaler`, fit 2D/3D PCA components, and visualise them interactively.

In [ ]:
TARGET_VARIABLE = "sepsis"

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

target = df_merged[TARGET_VARIABLE]
data = df_merged.drop(columns= [TARGET_VARIABLE])

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data)

pca = PCA(n_components=2)
pca_data = pca.fit_transform(scaled_data)

explained_variance = pca.explained_variance_ratio_
print(f"Varianza explicada por cada componente: {explained_variance}")
print(f"Varianza total explicada: {sum(explained_variance)}")

plt.figure(figsize=(8, 6))
plt.scatter(pca_data[:, 0], pca_data[:, 1], c=target, cmap='viridis', alpha=0.7)
plt.title("PCA 2D")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid()
plt.show()


In [ ]:
import plotly.express as px 

pca_3d = PCA(n_components=3)
pca_data_3d = pca_3d.fit_transform(scaled_data)

pca_df = pd.DataFrame(pca_data_3d, columns=["PC1", "PC2" ,"PC3"])
pca_df[TARGET_VARIABLE] = target

fig = px.scatter_3d(
    pca_df,
    x="PC1",
    y="PC2",
    z="PC3",
    color = TARGET_VARIABLE,
    title = "PCA - 3D Visualization",
    labels= {TARGET_VARIABLE},
    color_continuous_scale="Viridis", 
    opacity=0.7  
)
fig.update_traces(marker=dict(size=5))  
fig.update_layout(scene=dict(
    xaxis_title="PC 1",
    yaxis_title="PC 2",
    zaxis_title="PC 3"
))
fig.show()

In [ ]:
processed_df_copy = df_merged

target_copy = processed_df_copy[TARGET_VARIABLE]
scaler = MinMaxScaler()
X_preprocessed = pd.DataFrame(scaler.fit_transform(processed_df_copy.drop(columns=[TARGET_VARIABLE])))
X_preprocessed[TARGET_VARIABLE] = target_copy

target_palette = {0: "blue", 1: "red"}
row_colors = X_preprocessed[TARGET_VARIABLE].map(target_palette)
X_preprocessed = X_preprocessed.dropna() 
sns.clustermap(
    X_preprocessed.drop(columns=[TARGET_VARIABLE]), 
    cmap="coolwarm",
    row_colors=row_colors,
    figsize=(30, 60),
    annot=False,
    col_cluster=False
)

plt.title("Heatmap con Clustering y Anotación por Target", pad=100)
plt.show()